In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# fedavg
# df = pd.read_csv('/home1/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.04_23.22.30_fedavg/backdoor_tracking_log_nodefense.csv')
# flame
# df = pd.read_csv('/home1/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.04_23.24.42_flame/backdoor_tracking_log_nodefense.csv')
# krum
df = pd.read_csv('/home1/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.04_23.25.13_krum/backdoor_tracking_log_nodefense.csv')
# normbound
# df = pd.read_csv('/home1/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.04_23.25.29_normbound/backdoor_tracking_log_nodefense.csv')

# Create the plot
plt.figure(figsize=(12, 6))

# Plot each ASR column
plt.plot(df['iteration'], df['R1_ASR'], label='R1_ASR', marker='o', markersize=3)
plt.plot(df['iteration'], df['R2_ASR'], label='R2_ASR', marker='o', markersize=3)
plt.plot(df['iteration'], df['R3_ASR'], label='R3_ASR', marker='o', markersize=3)
plt.plot(df['iteration'], df['R4_ASR'], label='R4_ASR', marker='o', markersize=3)

# Add labels and title
plt.xlabel('Iteration')
plt.ylabel('ASR Value')
plt.title('ASR Values Across Iterations')
plt.legend()

# Add grid
plt.grid(True, linestyle='--', alpha=0.7)

# Show the plot
plt.tight_layout()
plt.show()

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path

# ---------------------------
# Small utilities
# ---------------------------

def get_region_colors():
    return {
        'R1': '#DAA520',  # Yellow
        'R2': '#EA6B66',  # Red
        'R3': '#2E8B57',  # Green
        'R4': '#1E90FF'   # Blue
    }

def parse_aggregation_rule(file_path: str) -> str:
    m = re.search(r'/([^/]+)/backdoor_tracking_log', file_path)
    return m.group(1) if m else "Unknown"

def load_and_prepare_df(file_path: str) -> pd.DataFrame:
    df = pd.read_csv(file_path)
    # numeric coercion
    for i in range(1, 5):
        asr_col = f'R{i}_ASR'
        sel_col = f'R{i}_selected'
        if asr_col in df.columns:
            df[asr_col] = pd.to_numeric(df[asr_col], errors='coerce').fillna(0.0)
        if sel_col in df.columns:
            df[sel_col] = pd.to_numeric(df[sel_col], errors='coerce').fillna(0).astype(int)
    if 'acc' in df.columns:
        df['acc'] = pd.to_numeric(df['acc'], errors='coerce')
    # order by iteration
    if 'iteration' in df.columns:
        df = df.sort_values('iteration').reset_index(drop=True)
    else:
        df['iteration'] = range(len(df))
    return df

def compute_signed_delta_asr(df: pd.DataFrame) -> pd.DataFrame:
    """
    ΔASR_t = ASR_t - ASR_{t-1} for each region on rounds when that region is selected.
    For the first-ever time a region is selected, we use baseline 0 (so ΔASR = ASR_t - 0).
    """
    records = []
    seen = {f'R{i}': False for i in range(1, 5)}

    for t in range(len(df)):
        it = int(df.loc[t, 'iteration'])
        for i in range(1, 5):
            reg = f'R{i}'
            sel_col = f'{reg}_selected'
            asr_col = f'{reg}_ASR'
            if sel_col not in df.columns or asr_col not in df.columns:
                continue
            if df.loc[t, sel_col] == 1:
                cur = float(df.loc[t, asr_col])
                if not seen[reg]:
                    prev = 0.0
                    seen[reg] = True
                else:
                    prev = float(df.loc[t-1, asr_col]) if t > 0 else 0.0
                records.append({'Region': reg, 'Iteration': it, 'Delta_ASR': cur - prev})
    return pd.DataFrame(records)

def melt_asr_series(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return long-form ASR dataframe: Region, Iteration, ASR, Selected (0/1).
    """
    rows = []
    for t in range(len(df)):
        it = int(df.loc[t, 'iteration'])
        for i in range(1, 5):
            reg = f'R{i}'
            asr_col = f'{reg}_ASR'
            sel_col = f'{reg}_selected'
            if asr_col not in df.columns:
                continue
            asr = float(df.loc[t, asr_col])
            sel = int(df.loc[t, sel_col]) if sel_col in df.columns else 0
            rows.append({'Region': reg, 'Iteration': it, 'ASR': asr, 'Selected': sel})
    return pd.DataFrame(rows)

def compute_prev_attack_flags(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each region & iteration t, mark PrevAttacked=1 if that region was selected at t-1.
    """
    out = []
    for t in range(len(df)):
        it = int(df.loc[t, 'iteration'])
        for i in range(1, 5):
            reg = f'R{i}'
            asr_col = f'{reg}_ASR'
            sel_prev_col = f'{reg}_selected'
            if asr_col not in df.columns:
                continue
            asr_t = float(df.loc[t, asr_col])
            prev_attacked = 0
            if t > 0 and sel_prev_col in df.columns:
                prev_attacked = int(df.loc[t-1, sel_prev_col])
            out.append({'Region': reg, 'Iteration': it, 'ASR': asr_t, 'PrevAttacked': prev_attacked})
    return pd.DataFrame(out)


# ---------------------------
# Plots
# ---------------------------

def plot_delta_asr_over_iterations(delta_df: pd.DataFrame, out_path: Path, title_prefix: str):
    if delta_df.empty:
        return
    colors = get_region_colors()
    plt.figure(figsize=(10, 5))
    sns.lineplot(
        data=delta_df,
        x='Iteration',
        y='Delta_ASR',
        hue='Region',
        hue_order=['R1', 'R2', 'R3', 'R4'],
        palette=colors,
        marker='o'
    )
    plt.axhline(0.0, color='gray', linewidth=1, linestyle='--')
    plt.title(f'{title_prefix}: ΔASR Over Iterations (signed)')
    plt.xlabel('Iteration')
    plt.ylabel('ΔASR (signed)')
    plt.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()

def plot_acc_and_asr(df: pd.DataFrame, out_path: Path, title_prefix: str):
    """
    Left y-axis: Accuracy; Right y-axis: ASR for each region.
    Highlight points where a region was attacked in the **previous iteration**.
    """
    if 'acc' not in df.columns or df['acc'].isna().all():
        return

    colors = get_region_colors()
    iters = df['iteration'].to_numpy()

    # right-axis data
    asr_long = melt_asr_series(df)
    prev_flags = compute_prev_attack_flags(df)  # Region, Iteration, ASR, PrevAttacked
    asr_long = asr_long.merge(prev_flags[['Region', 'Iteration', 'PrevAttacked']],
                              on=['Region', 'Iteration'], how='left')

    fig, ax_acc = plt.subplots(figsize=(12, 6))
    # Accuracy line (left axis)
    ax_acc.plot(iters, df['acc'], label='Accuracy', marker='o', linewidth=2, color='black')
    ax_acc.set_xlabel('Iteration')
    ax_acc.set_ylabel('Accuracy', color='black')
    ax_acc.tick_params(axis='y', labelcolor='black')

    # ASR lines (right axis)
    ax_asr = ax_acc.twinx()
    for reg in ['R1', 'R2', 'R3', 'R4']:
        sub = asr_long[asr_long['Region'] == reg]
        if sub.empty:
            continue
        ax_asr.plot(sub['Iteration'], sub['ASR'], label=f'{reg} ASR',
                    marker=None, linewidth=1.8, color=colors[reg], alpha=0.8)
        # highlight points where PrevAttacked=1
        hi = sub[sub['PrevAttacked'] == 1]
        ax_asr.scatter(hi['Iteration'], hi['ASR'],
                       label=f'{reg} (prev attacked)',
                       color=colors[reg], marker='X', s=60, edgecolors='white', linewidths=0.8)

    ax_asr.set_ylabel('ASR', color='dimgray')
    ax_asr.tick_params(axis='y', labelcolor='dimgray')

    # build a combined legend (avoid duplicates)
    handles, labels = [], []
    for ax in (ax_acc, ax_asr):
        h, l = ax.get_legend_handles_labels()
        for hh, ll in zip(h, l):
            if ll not in labels:
                handles.append(hh); labels.append(ll)
    plt.legend(handles, labels, loc='best', frameon=True)

    plt.title(f'{title_prefix}: Accuracy + ASR Over Iterations\n'
              f"(markers 'X' show rounds where the region was attacked in the previous iteration)")
    ax_acc.grid(True, axis='x', alpha=0.2); ax_acc.grid(True, axis='y', alpha=0.2)
    fig.tight_layout()
    fig.savefig(out_path)
    plt.close(fig)

def plot_box_delta_asr(delta_df: pd.DataFrame, out_path: Path, title_prefix: str):
    if delta_df.empty:
        return
    colors = get_region_colors()
    plt.figure(figsize=(8, 5))
    ax = sns.boxplot(
        data=delta_df,
        x='Region',
        y='Delta_ASR',
        order=['R1', 'R2', 'R3', 'R4'],
        palette=colors
    )
    # Explain boxplot in title/annotation
    plt.title(f'{title_prefix}: Distribution of ΔASR by Region\n'
              '(Box = IQR; center line = median; whiskers ≈ 1.5×IQR; circles = outliers)')
    plt.xlabel('Region'); plt.ylabel('ΔASR (signed)')
    plt.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()

def plot_box_asr(asr_df: pd.DataFrame, out_path: Path, title_prefix: str, attacked_only: bool = False):
    if asr_df.empty:
        return
    colors = get_region_colors()
    data = asr_df.copy()
    if attacked_only:
        data = data[data['Selected'] == 1]
    plt.figure(figsize=(8, 5))
    sns.boxplot(
        data=data,
        x='Region',
        y='ASR',
        order=['R1', 'R2', 'R3', 'R4'],
        palette=colors
    )
    tag = " (attacked rounds only)" if attacked_only else " (all rounds)"
    plt.title(f'{title_prefix}: Distribution of ASR by Region{tag}\n'
              '(Box = IQR; center line = median; whiskers ≈ 1.5×IQR; circles = outliers)')
    plt.xlabel('Region'); plt.ylabel('ASR')
    plt.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()


# ---------------------------
# Orchestrator
# ---------------------------

def analyze_asr_visuals(file_path, output_dir='plots'):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df = load_and_prepare_df(file_path)
    colors = get_region_colors()
    agg_rule = parse_aggregation_rule(file_path)

    # Describe selection method
    sel_cols = [f'R{i}_selected' for i in range(1, 5) if f'R{i}_selected' in df.columns]
    selected_counts = df[sel_cols].sum(axis=1) if sel_cols else pd.Series([0]*len(df))
    if (selected_counts == 1).all():
        selection_method = "Single Region Selection"
    elif (selected_counts == 2).all():
        selection_method = "Two Region Selection"
    else:
        selection_method = "Mixed Region Selection"
    title_prefix = f'[{agg_rule} | {selection_method}]'

    # Data transforms
    delta_df = compute_signed_delta_asr(df)
    asr_long = melt_asr_series(df)

    # PLOT 1: signed ΔASR over iterations
    plot_delta_asr_over_iterations(
        delta_df,
        out_path=output_dir / f"{agg_rule}_delta_asr_over_iterations_signed.png",
        title_prefix=title_prefix
    )

    # PLOT 2: Accuracy + ASR with “prev attacked” highlights
    plot_acc_and_asr(
        df,
        out_path=output_dir / f"{agg_rule}_acc_and_asr_over_iterations.png",
        title_prefix=title_prefix
    )

    # PLOT 3a: Boxplot of signed ΔASR
    plot_box_delta_asr(
        delta_df,
        out_path=output_dir / f"{agg_rule}_delta_asr_distribution_by_region_signed.png",
        title_prefix=title_prefix
    )

    # PLOT 3b: Boxplot of ASR (attacked-only)
    plot_box_asr(
        asr_long,
        out_path=output_dir / f"{agg_rule}_asr_distribution_by_region_attacked_only.png",
        title_prefix=title_prefix,
        attacked_only=True
    )

    # (Optional) PLOT 3c: Boxplot of ASR (all rounds)
    plot_box_asr(
        asr_long,
        out_path=output_dir / f"{agg_rule}_asr_distribution_by_region_all_rounds.png",
        title_prefix=title_prefix,
        attacked_only=False
    )


In [13]:
# single region selection
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.05_19.42.58_fedavg/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.05_19.45.30_flame/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.05_19.45.36_krum/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.05_19.45.42_normbound/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.05_21.26.16_geo/backdoor_tracking_log_nodefense.csv", output_dir="plots")
analyze_asr_visuals("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.11_23.12.30_fltrust/backdoor_tracking_log_nodefense.csv")

/tmp/ipykernel_1253210/1563006213.py:196: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.boxplot(
/tmp/ipykernel_1253210/1563006213.py:220: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_1253210/1563006213.py:220: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


In [ ]:
# double region selection
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.05_21.26.59_geo/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.06_20.09.11_fedavg/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.06_20.09.11_flame/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.06_20.09.21_krum/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.06_20.09.51_median/backdoor_tracking_log_nodefense.csv")
# analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.06_20.10.05_normbound/backdoor_tracking_log_nodefense.csv")
analyze_asr_increase("/home2/liyang/GitHub/Mirage/Mirage/saved_logs/_Aug.06_22.18.27_fltrust/backdoor_tracking_log_nodefense.csv")

/tmp/ipykernel_4014955/812404567.py:87: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
